In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
def analyze_tensile(filepath):
    """
    读取拉伸试验CSV，自动计算四项材料性能
    返回：结果字典 + 原始数据（用于画图）
    """
    # 1. 读取数据
    df = pd.read_csv(filepath)
    strain = df['应变'].values      # 应变数组（无量纲）
    stress = df['应力_MPa'].values  # 应力数组（MPa）
    name = df['材料'].iloc[0]       # 材料名字
    
    # 2. 弹性模量 E
    # 取前15%的数据点（弹性段），做线性拟合 y = kx + b，斜率k就是E
    n = len(strain)
    n_fit = max(20, int(n * 0.15))
    coeffs = np.polyfit(strain[:n_fit], stress[:n_fit], 1)
    E = coeffs[0]  # 单位：MPa
    
    # 3. 抗拉强度 σb：应力最大值
    tensile_strength = stress.max()
    
    # 4. 断裂延伸率 δ：最后一个应变值 × 100%
    fracture_strain = strain[-1] * 100  # 转成百分比
    
    # 5. 屈服强度 σs（0.2%残余应变法 —— 工程标准方法）
    # 原理：从应变=0.002处作一条平行于弹性段的直线
    #       这条线与应力-应变曲线的交点就是屈服强度
    offset_strain = 0.002
    offset_line = E * (strain - offset_strain)  # 偏移线方程
    
    # 找曲线与偏移线的交点（曲线从上方穿过偏移线）
    diff = stress - offset_line
    yield_strength = None
    
    for i in range(10, len(strain) - 1):
        if diff[i] > 0 and diff[i+1] <= 0:
            # 线性插值，算出更精确的交点应力
            ratio = diff[i] / (diff[i] - diff[i+1])
            yield_strength = stress[i] + ratio * (stress[i+1] - stress[i])
            yield_idx = i
            break
    
    # 如果没找到交点（比如某些高强化材料），fallback到0.5%应变处
    if yield_strength is None:
        idx = np.argmin(np.abs(strain - 0.005))
        yield_strength = stress[idx]
        yield_idx = idx
    
    return {
        '材料': name,
        '弹性模量E_MPa': round(E, 1),
        '屈服强度σs_MPa': round(yield_strength, 1),
        '抗拉强度σb_MPa': round(tensile_strength, 1),
        '断裂延伸率δ_%': round(fracture_strain, 1),
        'yield_idx': yield_idx  # 屈服点在数组中的位置，画图用
    }, strain, stress, offset_line

In [ ]:
# 确保这3个文件在你左侧文件栏里
files = ['Q235_tensile.csv', '45钢_tensile.csv', '6061铝_tensile.csv']

results = []

# 创建左右分栏：左边画图，右边画参数表
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for f in files:
    # 调用分析函数
    result, strain, stress, offset_line = analyze_tensile(f)
    results.append(result)
    
    # 左图：画应力-应变曲线
    axes[0].plot(strain, stress, label=result['材料'], linewidth=2)
    
    # 画出0.2%偏移线（只画到交点后一点点，避免太乱）
    idx = result['yield_idx']
    axes[0].plot(strain[:idx+20], offset_line[:idx+20], 
                '--', alpha=0.4, color='gray', linewidth=1)
    
    # 在屈服点位置画个圆圈标记
    axes[0].scatter(strain[idx], result['屈服强度σs_MPa'], 
                   s=100, zorder=5, edgecolors='black', facecolors='none', linewidth=1.5)

# 左图美化
axes[0].set_xlabel('Strain ε (mm/mm)', fontsize=12)
axes[0].set_ylabel('Stress σ (MPa)', fontsize=12)
axes[0].set_title('Stress-Strain Curves (0.2% Offset Method)', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(True, linestyle='--', alpha=0.5)
axes[0].set_xlim(left=0)

# 右图：用 matplotlib 画一个漂亮的参数对比表
axes[1].axis('off')  # 隐藏坐标轴

table_data = []
for r in results:
    table_data.append([
        r['材料'],
        f"{r['弹性模量E_MPa']:,.0f}",
        r['屈服强度σs_MPa'],
        r['抗拉强度σb_MPa'],
        r['断裂延伸率δ_%']
    ])

table = axes[1].table(
    cellText=table_data,
    colLabels=['Material', 'E (MPa)', 'σs (MPa)', 'σb (MPa)', 'δ (%)'],
    loc='center',
    cellLoc='center',
    colColours=['#4472C4']*5,
    colWidths=[0.2, 0.22, 0.2, 0.2, 0.18]
)

table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 2.2)

# 表头文字设为白色加粗
for i in range(5):
    table[(0, i)].set_text_props(color='white', fontweight='bold')

plt.tight_layout()
plt.show()

# 在终端打印文字版报告
print("\n" + "="*60)
print("           材料拉伸性能自动分析结果")
print("="*60)
for r in results:
    print(f"\n【{r['材料']}】")
    print(f"  弹性模量 E   = {r['弹性模量E_MPa']:>12,.1f} MPa")
    print(f"  屈服强度 σs  = {r['屈服强度σs_MPa']:>12,.1f} MPa")
    print(f"  抗拉强度 σb  = {r['抗拉强度σb_MPa']:>12,.1f} MPa")
    print(f"  断裂延伸率 δ = {r['断裂延伸率δ_%']:>12,.1f} %")
print("\n" + "="*60)